# Corporacion Favorita - New Superb Forecasting Model - 

## Imputation, Aggregation and Train-test-validation-split pipeline

#codi

Made by 4B Consultancy (Janne Heuvelmans, Georgi Duev, Alexander Engelage, Sebastiaan de Bruin) - 2024

In this data pipeline, 

The following steps are made within this notebook:  

>-0. Import Packages 

>-1. Load final dataset and aggregate dataset to weekly level
    -1.1 Load final dataset made in Data Preperation Pipeline Notebook
    -1.2 Aggregate dataset to weekly level

>-2. Column transformers and Train, Test, Validation Split

## 0. Import Packages

In [1]:
# Importing the libraries
import pandas as pd
import numpy as np
import polars as pl
import os
import sys
import altair as alt
import vegafusion as vf
import sklearn
import time
from datetime import date, datetime, timedelta
from sklearn.pipeline import Pipeline, make_pipeline

In [2]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.metrics import mean_absolute_percentage_error

import statsmodels.api as sm

## 1. Load final dataset

### 1.1. Functions - Import raw data from local PATH
Create import data function and give basic information function within the importing function.

Return basic information on each dataframe:  
- a) Information on the number of observation and features.  
- b) Information on the size of the dataframe. 

In [3]:
def f_get_data_and_info(import_path, file_name):

    print(f"\nReading file {file_name}\n")

    # Load data.
    df = pd.read_parquet(import_path + file_name + ".parquet")

    # Getting the basic information of the dataframe (number of observations and features, and size)
    print(
        f"The '{file_name}' dataframe contains: {df.shape[0]:,}".replace(",", ".")
        + f" observations and {df.shape[1]} features."
    )
    print(
        f"Prepared and transformed dataframe has optimized size of {round(sys.getsizeof(df)/1024/1024/1024, 2)} GB."
    )

    return df

### 1.2. Importing raw data
Importing parquet files with importing function (giving basic information)

In [ ]:
import_path = "C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/"

# Importing final df
df_final = f_get_data_and_info(import_path, file_name="Prepped_data_20241211")

# df_final = f_get_data_and_info(import_path, file_name="df_test_with_forecasts_20241120")

In [5]:
def get_columns(df):
    return df.columns.tolist()

In [ ]:
get_columns(df_final)

In [ ]:
df_final.info()

## 2.0 Train-test-validation-split

In [8]:
def train_test_val_split(df, window_length=26):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "week_number_cum"])

    # Get the maximum week in the dataset
    max_week = df["week_number_cum"].max()

    # Calculate start and end weeks for test and validation  sets
    val_week_end = max_week - 1

    val_week_start = max_week - window_length

    test_week_start = max_week - 2 * window_length

    test_week_end = val_week_start - 1

    train_week_start = max_week - 6 * window_length

    train_week_end = test_week_start - 1

    # Train data: All data before the start of the validation period
    train = df[
        (df["week_number_cum"] >= train_week_start)
        & (df["week_number_cum"] <= train_week_end)
    ]

    # Val data: From val_week_start to val_week_end
    test = df[
        (df["week_number_cum"] >= test_week_start)
        & (df["week_number_cum"] <= test_week_end)
    ]

    # Test data: From test_week_start to max_week
    val = df[
        (df["week_number_cum"] >= val_week_start)
        & (df["week_number_cum"] <= val_week_end)
    ]

    # Function to print split information
    def print_split_info(split_name, split):
        print(f"\n{split_name} set: shape: {split.shape}")
        print(f"{split_name} Min Week: {split['week_number_cum'].min()}")
        print(f"{split_name} Min Date: {split['date'].min()}")
        print(f"{split_name} Max Week: {split['week_number_cum'].max()}")
        print(f"{split_name} Max Date: {split['date'].max()}")
        print(f"{split_name} number of weeks: {split['week_number_cum'].nunique()}")
        print(f"Number of stores: {split['store_nbr'].nunique()}")
        print(f"Number of items: {split['item_nbr'].nunique()}")
        print(f"Size of {round(sys.getsizeof(split)/1024/1024/1024, 2)} GB.")

    # Print information about the splits
    print_split_info("Train", train)
    print_split_info("Test", test)
    print_split_info("Validation", val)

    return train, test, val

In [9]:
# train, test, val = train_test_val_split(df_final, window_length=26)

## 3.0 Functions - Impute stockouts and Aggregate dataset to weekly level


#### 3.1. Impute stockouts

Stockout on store level

•      Perishable good: when there are missing values for two consecutive days for a given item per individual store 

•      Nonperishable goods: when there are missing values for 7 consecutive days for a given item and per individual store

•      Action: Impute with Rolling Mean with default window of 7 days 

------------------------------------

In [10]:
def impute_stockouts_polars(df_pandas, window_size=7):
    # Convert the input Pandas DataFrame to a Polars DataFrame
    df = pl.from_pandas(df_pandas)

    # Sort the DataFrame by store number, item number, and date for proper ordering
    df = df.sort(["store_nbr", "item_nbr", "date"])

    # Create a boolean column to indicate where 'unit_sales' is missing
    df = df.with_columns((pl.col("unit_sales").is_null()).alias("is_missing"))

    # Assign a group identifier to each segment of missing or non-missing values
    df = df.with_columns(
        (
            pl.col("is_missing").cast(pl.Int16)
            != pl.col("is_missing").cast(pl.Int16).shift(1)
        )
        .cast(pl.Int16)
        .cum_sum()
        .alias("missing_group")
        .cast(pl.Int32)
    )

    # Calculate cumulative count of missing values within each segment of missing data
    df = df.with_columns(
        pl.when(pl.col("is_missing"))
        .then(
            pl.col("is_missing")
            .cast(pl.Int16)
            .cum_sum()
            .over(["store_nbr", "item_nbr", "missing_group"])
        )
        .otherwise(0)
        .alias("missing_count")
    )

    # Identify groups to find maximum of the same missing_count group
    df = df.with_columns(
        pl.col("missing_count")
        .max()
        .over(["missing_group"])
        .alias("group_max_missing_count")
        .cast(pl.Int16)
    )

    # Function for rolling mean imputation
    def rolling_mean_imputation(df, window_size=7):
        # Add rolling mean column for grouped data
        df = df.with_columns(
            pl.col("unit_sales")
            .rolling_mean(window_size=window_size, min_periods=1)
            .over(["store_nbr", "item_nbr"])
            .shift(1)  # Shift to exclude current row
            .alias("unit_sales_rolling_mean")
        )

        # Replace nulls in the target column with the calculated rolling mean
        df = df.with_columns(
            pl.when(pl.col("unit_sales").is_null())
            .then(pl.col("unit_sales_rolling_mean"))
            .otherwise(pl.col("unit_sales"))
            .alias("unit_sales")
        )

        # Drop the temporary rolling mean column
        df = df.drop("unit_sales_rolling_mean")

        return df

    # Apply rolling mean imputation based on perishable status
    df = df.with_columns(
        [
            pl.when(pl.col("perishable") == 1)  # If the item is perishable
            .then(
                pl.when(pl.col("group_max_missing_count") == 1)  # 1 missing value
                .then(0)  # Impute with 0
                .when(
                    pl.col("group_max_missing_count") > 2
                )  # More than 2 missing values
                .then(0)  # Impute with 0
                .when(
                    pl.col("group_max_missing_count") == 2
                )  # Exactly 2 missing values
                .then(
                    rolling_mean_imputation(df, window_size=7)["unit_sales"]
                )  # Impute with rolling mean for 2 missing days
                .otherwise(pl.col("unit_sales"))  # Keep original value
            )
            .when(pl.col("perishable") == 0)  # If the item is not perishable
            .then(
                pl.when(
                    pl.col("group_max_missing_count") > 7
                )  # More than 7 missing values
                .then(0)  # Impute with 0
                .when(
                    pl.col("group_max_missing_count") <= 7
                )  # 7 or fewer missing values
                .then(
                    rolling_mean_imputation(df, window_size=7)["unit_sales"]
                )  # Impute with rolling mean for missing 7 or fewer days
                .otherwise(pl.col("unit_sales"))  # Keep original value
            )
            .otherwise(pl.col("unit_sales"))  # For other cases, keep the original value
            .alias("unit_sales")
        ]
    )

    df = df.drop(
        "is_missing", "missing_group", "missing_count", "group_max_missing_count"
    )

    # Convert Polars df back to Pandas df
    df = df.to_pandas()

    return df

### 3.2. Aggregate dataset to weekly level

- Group the DataFrame by store number, item number, year, and week_cum_number, then aggregate the columns
--> "unit_sales","onpromotion", "holiday_local_count","holiday_regional_count","holiday_national_count",


In [11]:
def aggregate_week(df):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "year", "week_nbr"])

    # Group by the specified columns and aggregate
    df = (
        df.groupby(
            [
                "store_nbr",
                "item_nbr",
                "year",
                "week_number_cum",  # Aggregating by week_number_cum
            ]
        )
        .agg(
            {
                "unit_sales": "sum",
                "onpromotion": "sum",
                "holiday_local_count": "sum",
                "holiday_regional_count": "sum",
                "holiday_national_count": "sum",
                "date": "first",  # Keep the first day of week, needed to run Timeseries models from SKtime
                "store_type": "first",  # Keep the first occurrence of store_type
                "store_cluster": "first",  # Keep the first occurrence of store_cluster
                "item_family": "first",  # Keep the first occurrence of item_family
                "item_class": "first",  # Keep the first occurrence of item_class
                "perishable": "first",  # Keep the first occurrence of perishable
                "store_status": "last",  # Keep the last occurrence of store_status
                "item_status": "last",  # Keep the last occurrence of item_status
            }
        )
        .reset_index()
    )

    return df

## 4. Pipeline and preprocessing

Splitting and preprocessing with imputation and aggregating to weekly data

In [12]:
features = [
    "store_nbr",
    "item_nbr",
    "date",
    "onpromotion",
    # "holiday_local_count",
    # "holiday_national_count",
    # "holiday_regional_count",
    "store_type",
    "store_cluster",
    "item_family",
    "item_class",
    "perishable",
    # "store_status",
    # "item_status",
    "year",
    "week_number_cum",
]

target_variable = ["unit_sales"]

In [13]:
def impute_agg_preprocessing(df, window_size=7):

    df = impute_stockouts_polars(df, window_size)

    df = aggregate_week(df)

    return df

In [14]:
def preprocess_split_filter(df, features, target_variable):

    # Splitting in train, test, validation split
    print(f"\nStep 1: Splitting in train, test, validation split")
    train_df, test_df, val_df = train_test_val_split(df)

    # Preprocessing with imputation and aggregating to weekly data
    print(f"\nStep 2: Preprocessing with imputation and aggregating to weekly data")
    train_df = impute_agg_preprocessing(train_df)
    test_df = impute_agg_preprocessing(test_df)
    val_df = impute_agg_preprocessing(val_df)

    # Filter spilts on needed feature and target variables
    print(f"\nStep 3: Filter spilts on needed feature and target variables")
    train_df = train_df[features + target_variable]
    test_df = test_df[features + target_variable]
    val_df = val_df[features + target_variable]

    # Ensure df's are sorted by store, item, and date for alignment
    print(f"\nStep 4: Ensure dfs are sorted by store, item, and date for alignment")
    train_df = train_df.sort_values(by=["store_nbr", "item_nbr", "date"])
    test_df = test_df.sort_values(by=["store_nbr", "item_nbr", "date"])
    val_df = val_df.sort_values(by=["store_nbr", "item_nbr", "date"])

    return train_df, test_df, val_df

In [ ]:
train_df, test_df, val_df = preprocess_split_filter(df_final, features, target_variable)

Export train, test and validation dataframes as parquet files

In [16]:
# Export train_df to a Parquet file
train_df.to_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/train_df_20241211.parquet', index=False)

# Export test_df to a Parquet file
test_df.to_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/test_df_20241211.parquet', index=False)

# Export val_df to a Parquet file
val_df.to_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/val_df_20241211.parquet', index=False)
